## Cargue del Cubo de Datos (Oro)

**Taller ETL – Cubo SECOP**
**Autor:** Jurani Zabala Hernandez

**Objetivo:** Cargar en las tablas Hive del cubo (`secop_dw`, creadas en `CuboDatos.ipynb`) los datos normalizados del área **SILVER (plata)** (generados en `Transformacion.ipynb`), y validar el resultado con consultas SQL sobre `hecho_contratos` y sus 7 dimensiones — incluyendo la dimensión de rol `dim_tiempo`, usada 3 veces..

## 1. Sesión de Spark con soporte Hive

In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("SECOP_Cargue")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.hadoop.fs.defaultFS", "hdfs://namenode:9000")
    .config("spark.sql.catalogImplementation", "hive")
    .enableHiveSupport()
    .getOrCreate()
)

DB_NAME = "secop_dw"
spark.catalog.setCurrentDatabase(DB_NAME)
spark.sparkContext.setLogLevel("WARN")
print(f"✅ Sesión Spark inicializada. Base de datos activa: {DB_NAME}")

from pyspark.sql import SparkSession

print(f"✅ Sesión Spark inicializada. Base de datos activa: {DB_NAME}")


✅ Sesión Spark inicializada. Base de datos activa: secop_dw


## 2. Lectura del área Plata

In [3]:
SILVER_BASE = "hdfs://namenode:9000/datalake/silver/secop"

tablas = [
    "hecho_contratos", "dim_entidad", "dim_proveedor", "dim_tiempo",
    "dim_modalidad", "dim_ubicacion", "dim_estado_contrato", "dim_categoria",
]

dfs = {}
for tabla in tablas:
    ruta = f"{SILVER_BASE}/{tabla}"
    dfs[tabla] = spark.read.parquet(ruta)
    print(f"{tabla}: {dfs[tabla].count()} registros leídos desde {ruta}")

hecho_contratos: 18581 registros leídos desde hdfs://namenode:9000/datalake/silver/secop/hecho_contratos
dim_entidad: 2005 registros leídos desde hdfs://namenode:9000/datalake/silver/secop/dim_entidad
dim_proveedor: 18001 registros leídos desde hdfs://namenode:9000/datalake/silver/secop/dim_proveedor
dim_tiempo: 3255 registros leídos desde hdfs://namenode:9000/datalake/silver/secop/dim_tiempo
dim_modalidad: 14 registros leídos desde hdfs://namenode:9000/datalake/silver/secop/dim_modalidad
dim_ubicacion: 611 registros leídos desde hdfs://namenode:9000/datalake/silver/secop/dim_ubicacion
dim_estado_contrato: 9 registros leídos desde hdfs://namenode:9000/datalake/silver/secop/dim_estado_contrato
dim_categoria: 1471 registros leídos desde hdfs://namenode:9000/datalake/silver/secop/dim_categoria


## 3. Cargue en las tablas Hive del cubo

In [4]:
spark.conf.set("hive.exec.dynamic.partition", "true")
spark.conf.set("hive.exec.dynamic.partition.mode", "nonstrict")

dimensiones = [
    "dim_entidad", "dim_proveedor", "dim_tiempo",
    "dim_modalidad", "dim_ubicacion", "dim_estado_contrato", "dim_categoria",
]

for tabla in dimensiones:
    dfs[tabla].createOrReplaceTempView(f"tmp_{tabla}")
    spark.sql(f"INSERT OVERWRITE TABLE {DB_NAME}.{tabla} SELECT * FROM tmp_{tabla}")
    print(f"✅ {tabla} cargada en Hive.")

✅ dim_entidad cargada en Hive.


✅ dim_proveedor cargada en Hive.


✅ dim_tiempo cargada en Hive.


✅ dim_modalidad cargada en Hive.
✅ dim_ubicacion cargada en Hive.
✅ dim_estado_contrato cargada en Hive.


✅ dim_categoria cargada en Hive.


In [5]:
dfs["hecho_contratos"].createOrReplaceTempView("tmp_hecho_contratos")

columnas_hecho_sin_particion = [c for c in dfs["hecho_contratos"].columns if c != "anio_firma"]
select_cols = ", ".join(columnas_hecho_sin_particion)

spark.sql(f"""
INSERT OVERWRITE TABLE {DB_NAME}.hecho_contratos
    PARTITION (anio_firma)
SELECT {select_cols}, anio_firma
FROM tmp_hecho_contratos
""")

print("✅ hecho_contratos cargada en Hive (particionada por anio_firma).")

26/08/24 02:15:54 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.
26/08/24 02:15:54 WARN HiveConf: HiveConf of name hive.internal.ss.authz.settings.applied.marker does not exist
26/08/24 02:15:54 WARN HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
26/08/24 02:15:54 WARN HiveConf: HiveConf of name hive.stats.retries.wait does not exist
26/08/24 02:16:14 WARN log: Updating partition stats fast for: hecho_contratos  
26/08/24 02:16:14 WARN log: Updating partition stats fast for: hecho_contratos
26/08/24 02:16:14 WARN log: Updating partition stats fast for: hecho_contratos
26/08/24 02:16:14 WARN log: Updating partition stats fast for: hecho_contratos
26/08/24 02:16:14 WARN log: Updating partition stats fast for: hecho_contratos
26/08/24 02:16:14 WARN log: Updating partition stats fast for: hecho_contratos
26/08/24 02:16:14 WARN log: Updating partition stats fast for: hecho_co

✅ hecho_contratos cargada en Hive (particionada por anio_firma).


## 4. Validación del cargue: conteo de registros

In [6]:
todas_las_tablas = ["hecho_contratos"] + dimensiones
for tabla in todas_las_tablas:
    total = spark.sql(f"SELECT COUNT(*) AS total FROM {DB_NAME}.{tabla}").collect()[0]["total"]
    print(f"{tabla}: {total} registros en Hive")

hecho_contratos: 18581 registros en Hive
dim_entidad: 2005 registros en Hive
dim_proveedor: 18001 registros en Hive
dim_tiempo: 3255 registros en Hive
dim_modalidad: 14 registros en Hive
dim_ubicacion: 611 registros en Hive
dim_estado_contrato: 9 registros en Hive
dim_categoria: 1471 registros en Hive


## 5. Consultas de validación al cubo

Incluyen específicamente el uso de `dim_tiempo` en sus 3 roles distintos (firma, inicio, fin) mediante *joins* separados a la misma tabla física.

In [16]:
# Valor total contratado y cantidad de contratos por año de firma
spark.sql(f"""
SELECT anio_firma, COUNT(*) AS cantidad_contratos, format_number(SUM(valor_contrato), 2) AS valor_total
FROM {DB_NAME}.hecho_contratos
GROUP BY anio_firma
ORDER BY anio_firma
""").show()

+----------+------------------+------------------+
|anio_firma|cantidad_contratos|       valor_total|
+----------+------------------+------------------+
|      2016|                 6|    318,780,696.00|
|      2017|                79|  7,089,044,600.34|
|      2018|               472|132,852,538,760.01|
|      2019|               463|116,012,621,772.74|
|      2020|              1167|131,793,601,887.58|
|      2021|              1880|324,659,589,495.97|
|      2022|              2365|225,604,928,239.34|
|      2023|              2816|305,039,409,443.37|
|      2024|              3220|244,028,535,929.90|
|      2025|              3544|532,179,550,632.98|
|      2026|              2569|317,040,479,793.21|
+----------+------------------+------------------+



In [17]:
# Top 10 entidades por valor contratado (hecho + entidad)
spark.sql(f"""
SELECT e.nombre_entidad, e.sector, COUNT(*) AS cantidad_contratos, format_number(SUM(h.valor_contrato), 2) AS valor_total
FROM {DB_NAME}.hecho_contratos h
JOIN {DB_NAME}.dim_entidad e ON h.sk_entidad = e.sk_entidad
GROUP BY e.nombre_entidad, e.sector
ORDER BY valor_total DESC
LIMIT 10
""").show(truncate=False)

+----------------------------------------------------------------------------------------------------------------------------------+--------------------------------+------------------+--------------+
|nombre_entidad                                                                                                                    |sector                          |cantidad_contratos|valor_total   |
+----------------------------------------------------------------------------------------------------------------------------------+--------------------------------+------------------+--------------+
|PARQUES NACIONALES NATURALES DE COLOMBIA - DIRECCION TERRITORIAL CARIBE                                                           |Ambiente y Desarrollo Sostenible|8                 |99,831,459.00 |
|TRANSITO DE LOS PATIOS                                                                                                            |Transporte                      |7                 |99,144,000.00 |


In [18]:
# dim_tiempo en sus 3 ROLES: duración real (fin - inicio) por modalidad, usando 2 joins distintos a dim_tiempo
spark.sql(f"""
SELECT
    m.modalidad_contratacion,
    COUNT(*) AS cantidad_contratos,
    ROUND(AVG(DATEDIFF(t_fin.fecha, t_inicio.fecha)), 1) AS duracion_promedio_dias
FROM {DB_NAME}.hecho_contratos h
JOIN {DB_NAME}.dim_modalidad m     ON h.sk_modalidad = m.sk_modalidad
JOIN {DB_NAME}.dim_tiempo t_inicio ON h.sk_tiempo_inicio = t_inicio.sk_tiempo
JOIN {DB_NAME}.dim_tiempo t_fin    ON h.sk_tiempo_fin    = t_fin.sk_tiempo
GROUP BY m.modalidad_contratacion
ORDER BY duracion_promedio_dias DESC
""").show(truncate=False)

+-----------------------------------------------------------+------------------+----------------------+
|modalidad_contratacion                                     |cantidad_contratos|duracion_promedio_dias|
+-----------------------------------------------------------+------------------+----------------------+
|Licitación Pública Acuerdo Marco de Precios                |1                 |2248.0                |
|Concurso de méritos abierto                                |38                |543.3                 |
|Licitación pública Obra Publica                            |31                |391.8                 |
|Seleccion Abreviada Menor Cuantia Sin Manifestacion Interes|6                 |291.2                 |
|Licitación pública                                         |34                |285.0                 |
|Contratación Directa (con ofertas)                         |241               |280.1                 |
|Contratación régimen especial (con ofertas)                |154

In [11]:
# Distribución por ubicación y estado del contrato (hecho + ubicacion + estado)
spark.sql(f"""
SELECT u.departamento, es.estado_contrato, COUNT(*) AS cantidad_contratos, ROUND(SUM(h.valor_contrato), 2) AS valor_total
FROM {DB_NAME}.hecho_contratos h
JOIN {DB_NAME}.dim_ubicacion u        ON h.sk_ubicacion = u.sk_ubicacion
JOIN {DB_NAME}.dim_estado_contrato es ON h.sk_estado = es.sk_estado
GROUP BY u.departamento, es.estado_contrato
ORDER BY valor_total DESC
LIMIT 15
""").show(truncate=False)

+--------------------------+---------------+------------------+------------------+
|departamento              |estado_contrato|cantidad_contratos|valor_total       |
+--------------------------+---------------+------------------+------------------+
|Distrito Capital de Bogotá|Modificado     |1579              |5.5722178595106E11|
|Distrito Capital de Bogotá|En ejecución   |1590              |2.2621986298531E11|
|Distrito Capital de Bogotá|terminado      |653               |1.8876253369355E11|
|Antioquia                 |Modificado     |247               |1.3585824997273E11|
|Antioquia                 |En ejecución   |593               |1.233639269952E11 |
|Distrito Capital de Bogotá|Cerrado        |1960              |1.0326108707407E11|
|Antioquia                 |terminado      |278               |5.968389820572E10 |
|Atlántico                 |Modificado     |128               |5.899031065363E10 |
|Valle del Cauca           |terminado      |177               |4.174343972351E10 |
|Val

In [19]:
# Categoría del proceso + proveedor (hecho + categoria + proveedor)
spark.sql(f"""
SELECT c.codigo_categoria_principal, p.es_pyme, COUNT(*) AS cantidad_contratos, format_number(SUM(h.valor_contrato), 2) AS valor_total
FROM {DB_NAME}.hecho_contratos h
JOIN {DB_NAME}.dim_categoria c  ON h.sk_categoria = c.sk_categoria
JOIN {DB_NAME}.dim_proveedor p  ON h.sk_proveedor = p.sk_proveedor
GROUP BY c.codigo_categoria_principal, p.es_pyme
ORDER BY valor_total DESC
LIMIT 15
""").show(truncate=False)

+--------------------------+-------+------------------+--------------+
|codigo_categoria_principal|es_pyme|cantidad_contratos|valor_total   |
+--------------------------+-------+------------------+--------------+
|V1.92101602               |No     |4                 |994,164,331.00|
|V1.70131700               |No     |12                |992,966,390.00|
|V1.93131610               |No     |1                 |991,565,351.00|
|V1.39131700               |Si     |1                 |99,900,000.00 |
|V1.52152000               |Si     |1                 |99,841,642.00 |
|V1.44103103               |Si     |3                 |99,784,387.00 |
|V1.72102902               |No     |1                 |99,470,000.00 |
|V1.81141902               |No     |2                 |99,460,000.00 |
|V1.85121613               |No     |3                 |99,441,000.00 |
|V1.81101600               |No     |2                 |99,267,000.00 |
|V1.80141607               |Si     |9                 |985,769,700.00|
|V1.42

## Conclusiones

- Las 7 dimensiones y `hecho_contratos` fueron cargadas exitosamente desde plata, con el hecho particionado por `anio_firma`.
- Las consultas de validación confirman la integridad referencial del modelo, incluyendo el uso correcto de `dim_tiempo` como **dimensión de rol** (2 joins distintos a la misma tabla para calcular duración real de los contratos).
- Con esto se cierra el flujo ETL completo del taller.